# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MEDOXIII/FlyRank-Internship-ML/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

The dataset contains 30,000 pages. `content_age_days` ranges from 90 to 564 days, with a median of 236 days. `impressions_90d` is strongly right-skewed: its median is 731, while the maximum is 517,715, indicating that a small number of pages have exceptionally high impressions. The mean impression count is 5,200.37, which is much higher than the median because of a heavy upper tail. `avg_position = 0` represents missing position data rather than a real rank of zero. Rate fields such as CTR are stored as percentages multiplied by 100.

In [9]:
from pathlib import Path
import numpy as np
import pandas as pd

# Load the processed feature file
candidates = [
    Path("data/processed/refresh_feature_vector.csv"),
    Path("../../data/processed/refresh_feature_vector.csv"),
]

data_path = next((p for p in candidates if p.exists()), None)
if data_path is None:
    raise FileNotFoundError("Could not find refresh_feature_vector.csv")

df = pd.read_csv(data_path)

# Required columns for this audit
required = {
    "content_id",
    "content_age_days",
    "impressions_90d",
    "ctr",
    "avg_position",
    "trend_direction",
    "is_declining_label",
    "measurable_opportunity",
    "days_with_impressions",
    "sessions_90d",
}

missing = required - set(df.columns)
assert not missing, f"Missing columns: {sorted(missing)}"

# Sanity check: proxy label must match trend_direction == "down"
assert (
    df["is_declining_label"].astype(int)
    == df["trend_direction"].eq("down").astype(int)
).all(), "Label does not match trend_direction == 'down'"

# Use summary stats that are robust to skew and missing values
numeric_cols = [
    "content_age_days",
    "impressions_90d",
    "ctr",
    "avg_position",
    "days_since_last_update",
]

available = [c for c in numeric_cols if c in df.columns]
summary_df = df[available].replace({"avg_position": {0: np.nan}})

summary = summary_df.describe(percentiles=[0.25, 0.5, 0.75, 0.90, 0.99]).T
summary["skew"] = summary_df.skew(numeric_only=True)

display(summary.round(2))

,count,mean,std,min,25%,50%,75%,90%,99%,max,skew
content_age_days,30000.0,256.17,132.71,90.0,132.0,236.00,333.00,463.00,537.00,564.0,0.49
impressions_90d,30000.0,5200.37,16838.02,1.0,81.0,731.00,3615.25,12136.40,73505.83,517715.0,11.38
ctr,30000.0,0.51,3.28,0.0,0.0,0.07,0.29,0.65,8.33,100.0,17.44
avg_position,28795.0,17.03,15.15,0.1,6.7,11.40,22.90,37.50,70.50,245.0,2.01
days_since_last_update,30000.0,46.10,42.08,1.0,20.0,20.00,104.00,104.00,106.00,373.0,1.16


## 2. Signal test #1 / #2 / #3 (verdict each)

**Signal 1 — Content age: MIXED.** The observed decline-proxy rate rose from 0.600 in the youngest group to 0.640 in the second group, then fell to 0.494 and 0.422 in the older groups. The pattern is not monotonic, and the oldest group had a lower rate than the youngest group. Therefore, the data do not support a simple assumption that older pages consistently have higher decline-proxy rates.

**Signal 2 — Trailing-90-day impressions: MIXED.** The observed decline-proxy rate rose from 0.376 in the lowest-impression group to 0.605 and 0.626 in the middle groups, then fell to 0.562 in the highest group. Impressions show a non-monotonic relationship with the decline proxy rather than a consistently increasing pattern.

**Signal 3 — CTR: CONFIRMED for the tested directional association.** Among the 16,726 pages with at least 500 impressions and a known average position, the observed decline-proxy rate decreased across CTR groups, from 0.664 in the lowest-CTR group to 0.511 in the highest. This is a descriptive association with the current-window proxy label; it does not show that low CTR causes decline or predicts future decline. Average position was not controlled for, so position differences may partly explain the observed pattern.

In [10]:
def quartile_label_rates(frame, signal, label="is_declining_label"):
    temp = frame[[signal, label]].dropna().copy()
    unique_values = temp[signal].nunique()

    if unique_values < 2:
        raise ValueError(f"{signal} has fewer than two distinct values")

    q = min(4, unique_values)
    temp["quantile_bin"] = pd.qcut(
        temp[signal],
        q=q,
        duplicates="drop",
    )

    return temp.groupby("quantile_bin", observed=False)[label].agg(
        rows="size",
        decline_rate="mean",
    )

print("Signal 1: content age")
display(quartile_label_rates(df, "content_age_days").round(3))

print("Signal 2: trailing-90-day impressions")
display(quartile_label_rates(df, "impressions_90d").round(3))

visible_known_position = df.loc[
    df["impressions_90d"].ge(500)
    & df["avg_position"].gt(0)
    & df["ctr"].notna()
].copy()

print("Signal 3: CTR among visible pages with known average position")
display(quartile_label_rates(visible_known_position, "ctr").round(3))
print("Rows in the CTR test:", len(visible_known_position))

Signal 1: content age


,rows,decline_rate
quantile_bin,,
"(89.999, 132.0]",7518,0.600
"(132.0, 236.0]",8128,0.640
"(236.0, 333.0]",6917,0.494
"(333.0, 564.0]",7437,0.422


Signal 2: trailing-90-day impressions


,rows,decline_rate
quantile_bin,,
"(0.999, 81.0]",7503,0.376
"(81.0, 731.0]",7499,0.605
"(731.0, 3615.25]",7498,0.626
"(3615.25, 517715.0]",7500,0.562


Signal 3: CTR among visible pages with known average position


,rows,decline_rate
quantile_bin,,
"(-0.001, 0.07]",4550,0.664
"(0.07, 0.17]",4059,0.620
"(0.17, 0.35]",4065,0.579
"(0.35, 5.43]",4052,0.511


Rows in the CTR test: 16726


## 3. The flag-linked test

The flagged pages (`measurable_opportunity = 1`, n = 22,006) had much higher median impressions (1,704.5 vs. 12.0), more median days with impressions (87 vs. 8), more median sessions (14 vs. 2), and a higher observed decline-proxy rate (0.598 vs. 0.389) than unflagged pages (n = 7,994). The known-position rate was also higher for flagged pages (1.000 vs. 0.849), so the flag is associated with higher observed activity and greater measurement coverage in this snapshot. This is a descriptive comparison, not independent validation of the flag rule. If these fields are part of the flag construction, some of the observed differences are expected from the definition of the flag itself.

In [11]:
flag_summary = df.groupby("measurable_opportunity", dropna=False).agg(
    rows=("content_id", "size"),
    proxy_decline_rate=("is_declining_label", "mean"),
    median_impressions=("impressions_90d", "median"),
    median_days_with_impressions=("days_with_impressions", "median"),
    median_sessions=("sessions_90d", "median"),
    median_ctr=("ctr", "median"),
    known_position_rate=("avg_position", lambda s: s.gt(0).mean()),
).round(3)

display(flag_summary)

,rows,proxy_decline_rate,median_impressions,median_days_with_impressions,median_sessions,median_ctr,known_position_rate
measurable_opportunity,,,,,,,
0,7994,0.389,12.0,8.0,2.0,0.00,0.849
1,22006,0.598,1704.5,87.0,14.0,0.14,1.000


## 4. What this means in practice

In this snapshot, CTR showed the clearest directional association with the decline-proxy label, while age and impressions showed mixed patterns. Pages marked `measurable_opportunity` had substantially higher observed activity and a higher decline-proxy rate than unflagged pages, but this comparison does not independently validate the flag’s rule. These are descriptive, within-window findings for human-review decision support; they do not establish causes or predict future performance.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.